# K-Means + K-Nearest Neighbours from Scratch
Both algorithms implemented with NumPy only (plus matplotlib for plots).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## Part 1 — K-Means

In [ ]:
X = np.array([[1,1],[1.5,2],[3,4],[5,7],[3.5,5],[4.5,5],[3.5,4.5]])
init_centroids = np.array([[1.0, 1.0], [5.0, 7.0]])

**Core steps:** assign each point to its nearest centroid, then move each centroid to the mean of its points. Repeat until centroids stop moving.

In [ ]:
def assign_clusters(X, centroids):
    # distance from every point to every centroid -> (m, K); pick the closest
    return np.linalg.norm(X[:, None] - centroids[None], axis=2).argmin(axis=1)

def compute_centroids(X, idx, centroids):
    # keep the old centroid if a cluster ends up empty
    return np.array([X[idx == k].mean(axis=0) if np.any(idx == k) else centroids[k]
                     for k in range(len(centroids))])

def compute_cost(X, idx, centroids):
    # mean squared distance of points to their assigned centroid
    return np.mean(np.sum((X - centroids[idx]) ** 2, axis=1))

def run_kmeans(X, centroids, max_iters=100):
    for _ in range(max_iters):
        idx = assign_clusters(X, centroids)
        new = compute_centroids(X, idx, centroids)
        if np.allclose(new, centroids):
            break
        centroids = new
    return centroids, assign_clusters(X, centroids)

In [ ]:
centroids, idx = run_kmeans(X, init_centroids)
print("Final centroids:\n", centroids)
print("Labels:", idx)
print("Cost:", compute_cost(X, idx, centroids))

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=idx, s=60, cmap="coolwarm")
plt.scatter(*centroids.T, marker="X", color="black", s=200, label="Centroids")
plt.xlabel("X"); plt.ylabel("Y"); plt.legend(); plt.title("K-Means result"); plt.show()

### Elbow method
Run K-Means for several K and look for the 'bend' in the cost curve.

In [ ]:
ks = range(1, 5)
costs = []
for k in ks:
    c, i = run_kmeans(X, X[np.random.choice(len(X), k, replace=False)], 200)
    costs.append(compute_cost(X, i, c))
print(dict(zip(ks, np.round(costs, 3))))
plt.plot(ks, costs, marker="o"); plt.xlabel("K"); plt.ylabel("Cost"); plt.title("Elbow method"); plt.show()

## Part 2 — K-Nearest Neighbours

In [ ]:
X_train = np.array([[1,1],[1.5,1.2],[1.8,1.5],[2,1.8],[2.2,2],
                    [4,4],[4.2,4.5],[4.5,4.2],[5,5],[5.5,5.2]])
y_train = np.array([0]*5 + [1]*5)
X_test  = np.array([[2.8,2.9],[4.8,4.7],[1.2,1.1],[4.2,4.0]])
y_test  = np.array([0, 1, 0, 1])

for c in (0, 1):
    plt.scatter(*X_train[y_train == c].T, s=70, label=f"Class {c}")
plt.xlabel("Feature 1"); plt.ylabel("Feature 2"); plt.title("KNN training data"); plt.legend(); plt.show()

**Core steps:** compute distance to every training point, take the K nearest, and return the majority label.

In [ ]:
def knn_predict_one(X_train, y_train, point, K):
    nearest = np.argsort(np.linalg.norm(X_train - point, axis=1))[:K]
    return np.bincount(y_train[nearest]).argmax()

def knn_predict(X_train, y_train, X_test, K):
    return np.array([knn_predict_one(X_train, y_train, p, K) for p in X_test])

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

y_pred = knn_predict(X_train, y_train, X_test, K=3)
print("Predicted:", y_pred, "| Actual:", y_test, "| Accuracy:", accuracy(y_test, y_pred))

### Effect of K

In [ ]:
for K in (1, 3, 5, 7):
    print(f"K={K}  accuracy={accuracy(y_test, knn_predict(X_train, y_train, X_test, K)):.2f}")